# Aula 3 — Regressão Logística

Como usar este notebook: a **Parte A** tem células que o professor roda e
explica durante a aula. Acompanhe na tela, sem precisar digitar nada. A
**Parte B** é com você: complete os exercícios nos lugares marcados com
`# SEU CODIGO AQUI`.

Se ainda não sabe como abrir e salvar sua própria cópia deste notebook,
veja a página **Antes de começar** no material da aula antes de continuar.

## Parte A: Demonstração

### Os dados: 400 clientes de um serviço de streaming

A coluna `cancelou` vale 1 para quem cancelou e 0 para quem ficou. É essa
coluna que o modelo vai tentar prever.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Endereço dos dados desta aula no GitHub.
URL_DADOS = "https://raw.githubusercontent.com/klein-natan/nanodegree-AI-Atitus/main/data/assinaturas.csv"
# Alternativa para testar offline, antes do repositório existir no GitHub:
# URL_DADOS = "../../data/assinaturas.csv"

dados = pd.read_csv(URL_DADOS)
dados.head()

In [ ]:
taxa = dados["cancelou"].mean()
print(f"Cancelaram: {taxa * 100:.1f}% dos clientes")
print(f"Chutar 'ninguém cancela' já acerta {(1 - taxa) * 100:.1f}% das vezes")
print()
print(dados.groupby("chamados_suporte")["cancelou"].mean().round(2))

### A função sigmoide

É ela que transforma qualquer número em uma probabilidade entre 0 e 1:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

In [ ]:
def sigmoide(z):
    # Espreme qualquer numero na faixa de 0 a 1
    return 1 / (1 + np.exp(-z))

print(f"sigmoide(0)  = {sigmoide(0):.2f}")
print(f"sigmoide(2)  = {sigmoide(2):.2f}")
print(f"sigmoide(-2) = {sigmoide(-2):.2f}")

valores_z = np.arange(-8, 8, 0.1)
plt.plot(valores_z, sigmoide(valores_z))
plt.axhline(0.5)
plt.xlabel("z")
plt.ylabel("Probabilidade")
plt.title("A função sigmoide")
plt.show()

### O modelo logístico

A soma de sempre produz o $z$, e a sigmoide transforma esse $z$ em
probabilidade:

$$P(\text{cancelar}) = \sigma(z) \qquad z = w_0 + w_1 x_1 + \dots + w_p x_p$$

O plano vira colunas de 0 e 1, igual ao bairro da Aula 2.

In [ ]:
from sklearn.linear_model import LogisticRegression

colunas_do_modelo = ["meses_de_casa", "valor_mensal", "chamados_suporte", "plano"]
tabela_modelo = pd.get_dummies(dados[colunas_do_modelo], columns=["plano"], drop_first=True)
tabela_modelo = tabela_modelo.astype(float)

modelo = LogisticRegression(max_iter=1000)
modelo.fit(tabela_modelo, dados["cancelou"])

print(f"w0: {modelo.intercept_[0]:.3f}")
for nome, peso in zip(tabela_modelo.columns, modelo.coef_[0]):
    print(f"{nome}: {peso:.3f}")

### Lendo os coeficientes em chances

No mundo das chances, cada coeficiente vira uma multiplicação:

$$\frac{p}{1 - p} = e^{z}$$

Somar 1 na variável $x_j$ multiplica a chance por $e^{w_j}$.

In [ ]:
# Converte cada coeficiente em "por quanto a chance é multiplicada"
for nome, peso in zip(tabela_modelo.columns, modelo.coef_[0]):
    fator = np.exp(peso)
    print(f"{nome}: multiplica a chance por {fator:.2f}")

In [ ]:
# Dois clientes bem diferentes, com as colunas na mesma ordem do treino
cliente_em_risco = pd.DataFrame({
    "meses_de_casa": [6.0],
    "valor_mensal": [130.0],
    "chamados_suporte": [3.0],
    "plano_Padrão": [0.0],
    "plano_Premium": [1.0],
})
cliente_tranquilo = pd.DataFrame({
    "meses_de_casa": [36.0],
    "valor_mensal": [40.0],
    "chamados_suporte": [0.0],
    "plano_Padrão": [0.0],
    "plano_Premium": [0.0],
})

print(f"Cliente em risco:  {modelo.predict_proba(cliente_em_risco)[0][1]:.2f}")
print(f"Cliente tranquilo: {modelo.predict_proba(cliente_tranquilo)[0][1]:.2f}")

### Matriz de confusão e métricas

Com o limiar padrão de 0,5:

$$\text{acurácia} = \frac{VP + VN}{VP + VN + FP + FN}$$

$$\text{precisão} = \frac{VP}{VP + FP} \qquad \text{recall} = \frac{VP}{VP + FN}$$

In [ ]:
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score, f1_score,
    brier_score_loss,
)

probabilidades = modelo.predict_proba(tabela_modelo)[:, 1]
previsoes = (probabilidades >= 0.5).astype(int)

print(confusion_matrix(dados["cancelou"], previsoes))
print()
print(f"Acurácia: {accuracy_score(dados['cancelou'], previsoes):.2f}")
print(f"Precisão: {precision_score(dados['cancelou'], previsoes):.2f}")
print(f"Recall:   {recall_score(dados['cancelou'], previsoes):.2f}")
print(f"F1:       {f1_score(dados['cancelou'], previsoes):.2f}")

In [ ]:
# O mesmo modelo, com o limiar mais baixo: a lista de risco cresce
for limiar in [0.7, 0.5, 0.3]:
    previsoes_limiar = (probabilidades >= limiar).astype(int)
    precisao = precision_score(dados["cancelou"], previsoes_limiar)
    recall = recall_score(dados["cancelou"], previsoes_limiar)
    print(f"limiar {limiar}: lista de {previsoes_limiar.sum()} clientes, "
          f"precisão {precisao:.2f}, recall {recall:.2f}")

### A curva ROC: todos os limiares de uma vez

Cada limiar dá um par de números. Marque todos e você tem a curva:

$$\text{TPR} = \frac{VP}{VP + FN} \qquad \text{FPR} = \frac{FP}{FP + VN}$$

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

fpr, tpr, limiares_roc = roc_curve(dados["cancelou"], probabilidades)
auc = roc_auc_score(dados["cancelou"], probabilidades)

plt.figure(figsize=(6.5, 6))
plt.plot([0, 1], [0, 1], linestyle="--", label="chutar no acaso (AUC = 0,50)")
plt.plot(fpr, tpr, label=f"o nosso modelo (AUC = {auc:.2f})")
for alvo in [0.7, 0.5, 0.3]:
    i = int(np.argmin(np.abs(limiares_roc - alvo)))
    plt.scatter([fpr[i]], [tpr[i]])
    plt.annotate(f"limiar {alvo}", (fpr[i], tpr[i]), xytext=(8, -12),
                 textcoords="offset points")
plt.xlabel("FPR: dos que ficaram, quantos acusei à toa")
plt.ylabel("TPR: dos que saíram, quantos peguei")
plt.title("A curva ROC")
plt.legend()
plt.show()

print(f"AUC = {auc:.3f}")

### O que a AUC significa, conferido por sorteio

A AUC é a probabilidade de o modelo dar nota maior a um cliente que
cancelou do que a um que ficou. Isso não é analogia: dá para conferir.

In [ ]:
sorteio = np.random.default_rng(0)
notas_cancelou = probabilidades[dados["cancelou"] == 1]
notas_ficou = probabilidades[dados["cancelou"] == 0]

pares = 200_000
um_cancelou = sorteio.choice(notas_cancelou, pares)
um_ficou = sorteio.choice(notas_ficou, pares)
proporcao = (um_cancelou > um_ficou).mean()

quantos = f"{pares:,}".replace(",", ".")
print(f"Sorteamos {quantos} pares (um que cancelou, um que ficou).")
print(f"O que cancelou recebeu nota maior em {100 * proporcao:.1f}% das vezes.")
print(f"A AUC calculada pela fórmula é     {100 * auc:.1f}%.")

### Calibração: a probabilidade é honesta?

De cem clientes a quem o modelo deu 30%, quantos cancelam de fato? Se a
resposta for perto de trinta, o modelo é calibrado.

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

aconteceu, prometido = calibration_curve(dados["cancelou"], probabilidades,
                                         n_bins=8, strategy="quantile")

plt.figure(figsize=(6.5, 6))
plt.plot([0, 1], [0, 1], linestyle="--", label="o ideal")
plt.plot(prometido, aconteceu, "o-", label="o nosso modelo")
plt.xlabel("O que o modelo prometeu")
plt.ylabel("O que aconteceu de fato")
plt.title("Curva de calibração")
plt.legend()
plt.show()

for p_previsto, p_real in zip(prometido, aconteceu):
    print(f"  prometeu {100 * p_previsto:>5.1f}%  ->  aconteceu {100 * p_real:>5.1f}%")

print()
print(f"Brier: {brier_score_loss(dados['cancelou'], probabilidades):.3f}")

### O que quebra a calibração

`class_weight="balanced"` dá mais peso à classe minoritária. Repare no
que ele faz com a AUC e no que faz com a probabilidade.

In [ ]:
modelo_equilibrado = LogisticRegression(max_iter=1000, class_weight="balanced")
modelo_equilibrado.fit(tabela_modelo, dados["cancelou"])
prob_equilibrada = modelo_equilibrado.predict_proba(tabela_modelo)[:, 1]

print(f"{'':<14} {'AUC':>8} {'Brier':>8} {'prob. média':>13}")
for nome, prob in [("normal", probabilidades), ("balanced", prob_equilibrada)]:
    print(f"{nome:<14} {roc_auc_score(dados['cancelou'], prob):>8.4f} "
          f"{brier_score_loss(dados['cancelou'], prob):>8.3f} "
          f"{prob.mean():>13.3f}")
print(f"{'a verdade':<14} {'':>8} {'':>8} {dados['cancelou'].mean():>13.3f}")

print()
print("A AUC é praticamente a mesma: a ordenação não mudou.")
print("A probabilidade média saiu de 0,363 para 0,457, com a taxa real em")
print("0,362. Para ordenar uma lista, tanto faz. Para contar dinheiro, não.")

## Parte B: Exercícios

Complete cada exercício no espaço marcado com `# SEU CODIGO AQUI`. Rode a
célula de verificação logo depois para conferir sua resposta.

### Exercício 1: conhecendo os clientes

Rode a célula abaixo e observe: quantos meses de casa tem o cliente mais
antigo? Qual é o maior número de chamados ao suporte?

In [ ]:
dados.describe()

In [ ]:
if len(dados) == 400:
    print(f"✅ Os dados têm {len(dados)} clientes, como esperado.")
else:
    print("❌ Confira se você rodou a célula que carrega os dados, no início do notebook.")

### Exercício 2: quem cancela mais?

Rode a célula e observe a taxa de cancelamento por plano. O plano mais
caro é o que mais perde clientes?

In [ ]:
taxa_por_plano = dados.groupby("plano")["cancelou"].mean().round(3)
print(taxa_por_plano)

In [ ]:
print("Converse com um colega: por que o plano Premium cancela menos, se ele é o mais caro?")

### Exercício 3: treinando o classificador

Crie as colunas de plano com `get_dummies` e treine um
`LogisticRegression` chamado `meu_modelo`. Use `max_iter=1000`.

In [ ]:
colunas_escolhidas = ["meses_de_casa", "valor_mensal", "chamados_suporte", "plano"]
minha_tabela = pd.get_dummies(dados[colunas_escolhidas], columns=["plano"], drop_first=True)
minha_tabela = minha_tabela.astype(float)

In [ ]:
# SEU CODIGO AQUI

In [ ]:
for nome, peso in zip(minha_tabela.columns, meu_modelo.coef_[0]):
    print(f"{nome}: {peso:.3f}")

In [ ]:
if abs(meu_modelo.coef_[0][2] - 0.996) < 0.05:
    print("✅ O coeficiente dos chamados ficou perto de 1,00, como esperado.")
else:
    print("❌ Confira se minha_tabela tem as cinco colunas e o alvo é dados['cancelou'].")

### Exercício 4: de coeficiente para chance

Transforme o coeficiente dos chamados ao suporte no fator que multiplica
a chance, usando `np.exp`:

$$\text{fator} = e^{w_j}$$

In [ ]:
coeficiente_chamados = meu_modelo.coef_[0][2]

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(f"Cada chamado a mais multiplica a chance de cancelar por {fator_chamados:.2f}")

In [ ]:
if abs(fator_chamados - 2.71) < 0.2:
    print("✅ Perto de 2,7: cada chamado quase triplica a chance de cancelar.")
else:
    print("❌ Confira se você aplicou np.exp no coeficiente certo (o dos chamados).")

### Exercício 5: a matriz de confusão

Calcule as probabilidades com `predict_proba`, transforme em previsões
com o limiar de 0,5 e monte a matriz de confusão.

In [ ]:
minhas_probabilidades = meu_modelo.predict_proba(minha_tabela)[:, 1]

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(minha_matriz)
print(f"Acurácia: {accuracy_score(dados['cancelou'], minhas_previsoes):.2f}")
print(f"Precisão: {precision_score(dados['cancelou'], minhas_previsoes):.2f}")
print(f"Recall:   {recall_score(dados['cancelou'], minhas_previsoes):.2f}")

In [ ]:
if minha_matriz.sum() == 400 and minha_matriz[1][1] > 70:
    print("✅ A matriz soma 400 clientes e o modelo pegou mais de 70 cancelamentos.")
else:
    print("❌ Confira se você usou a coluna 1 do predict_proba (a de cancelar).")

### Exercício 6: mexendo no limiar

Refaça as previsões com o limiar de 0,30 e calcule o recall. Compare com
o recall do limiar de 0,50.

In [ ]:
limiar_novo = 0.30

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(f"Lista de risco com limiar {limiar_novo}: {previsoes_novas.sum()} clientes")
print(f"Recall com limiar {limiar_novo}: {recall_novo:.2f}")

In [ ]:
if recall_novo > 0.8:
    print("✅ O recall passou de 0,80: o modelo agora pega a maioria dos cancelamentos.")
else:
    print("❌ Esperava um recall bem mais alto. Confira o sinal da comparação (>=).")

### Exercício 7: a lista inteira de limiares

Rode o laço abaixo, que mostra precisão e recall para seis limiares.
Repare no tamanho da lista em cada linha. Os três exercícios seguintes
transformam essa tabela em decisão.

In [ ]:
for limiar in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    previsoes_teste = (minhas_probabilidades >= limiar).astype(int)
    precisao_teste = precision_score(dados["cancelou"], previsoes_teste)
    recall_teste = recall_score(dados["cancelou"], previsoes_teste)
    print(f"limiar {limiar}: lista de {previsoes_teste.sum():3d} clientes, "
          f"precisão {precisao_teste:.2f}, recall {recall_teste:.2f}")

In [ ]:
print("Não existe resposta única aqui: o limiar certo depende do custo de cada erro.")
print("O exercício 10 põe preço nos dois erros, e aí a resposta aparece sozinha.")

### Exercício 8: a curva ROC do seu modelo

Calcule `minha_auc` com `roc_auc_score` e desenhe a curva com
`roc_curve`, usando `minhas_probabilidades` do exercício 5.

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

In [ ]:
# SEU CODIGO AQUI

In [ ]:
plt.figure(figsize=(6, 5.5))
plt.plot([0, 1], [0, 1], linestyle="--")
plt.plot(meu_fpr, meu_tpr)
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title(f"AUC = {minha_auc:.3f}")
plt.show()

print(f"AUC: {minha_auc:.3f}")

In [ ]:
if 0.75 < minha_auc < 0.90:
    print(f"✅ AUC de {minha_auc:.2f}: sorteando um cliente que cancelou e um")
    print(f"   que ficou, o modelo acerta a ordem em {100 * minha_auc:.0f}% dos pares.")
else:
    print("❌ Confira se você passou as probabilidades, e não as previsões 0/1.")

### Exercício 9: a AUC enxerga a ordem, e só

Some 0,2 a todas as probabilidades (limitando em 1) e calcule a AUC de
novo. A ordem entre os clientes não muda. E a AUC?

In [ ]:
probabilidades_infladas = np.minimum(minhas_probabilidades + 0.2, 1.0)

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(f"AUC normal:   {minha_auc:.4f}      Brier normal:  {brier_normal:.3f}")
print(f"AUC inflada:  {auc_inflada:.4f}      Brier inflado: {brier_inflado:.3f}")

In [ ]:
if abs(auc_inflada - minha_auc) < 0.02 and brier_inflado > brier_normal:
    print("✅ A AUC quase não mudou, e o Brier piorou bastante.")
    print("   A AUC só olha a ordem. O Brier olha se o número é verdade.")
    print("   Um modelo pode ordenar bem e mentir nas probabilidades.")
else:
    print("❌ Confira se você usou brier_score_loss nas duas versões.")

### Exercício 10: desafio, o limiar que custa menos

Agora a decisão de negócio, com número. O cupom de retenção custa R\\$ 20
e perder um cliente custa R\\$ 300:

$$\text{custo} = FP \cdot 20 + FN \cdot 300$$

Percorra os limiares de 0,01 a 0,99 e ache o de menor custo.

In [ ]:
CUSTO_FP = 20
CUSTO_FN = 300
limiares_teste = np.arange(0.01, 1.00, 0.01)


def em_reais(valor):
    """R$ 3.900, com ponto de milhar, do jeito que se escreve em português."""
    return "R$ " + f"{valor:,}".replace(",", ".")

In [ ]:
# SEU CODIGO AQUI

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(limiares_teste, custos)
plt.scatter([melhor_limiar], [custos.min()])
plt.xlabel("Limiar")
plt.ylabel("Custo total dos erros (R$)")
plt.title("O custo de cada limiar")
plt.show()

lista_melhor = int((minhas_probabilidades >= melhor_limiar).sum())
custo_meio = custos[int(np.argmin(np.abs(limiares_teste - 0.5)))]
print(f"Melhor limiar: {melhor_limiar:.2f}, custo {em_reais(custos.min())}, "
      f"lista de {lista_melhor} clientes")
print(f"Limiar padrão 0,50: custo {em_reais(custo_meio)}")
print(f"Economia: {em_reais(custo_meio - custos.min())}")
print()
print(f"Ligar para todo mundo custaria {em_reais((dados['cancelou'] == 0).sum() * CUSTO_FP)}")
print(f"Não ligar para ninguém custaria {em_reais(dados['cancelou'].sum() * CUSTO_FN)}")

In [ ]:
teorico = CUSTO_FP / (CUSTO_FP + CUSTO_FN)
if melhor_limiar < 0.25 and custos.min() < custo_meio:
    print(f"✅ O melhor limiar ({melhor_limiar:.2f}) é bem menor que 0,50.")
    print(f"   A conta fechada dá {teorico:.2f}, e a curva é plana nessa região:")
    print("   qualquer limiar por ali custa quase o mesmo.")
    print("   Repare que ligar para todos já é quase tão bom: quando um erro")
    print("   custa 15 vezes o outro, o modelo agrega pouco.")
else:
    print("❌ Confira a ordem do confusion_matrix.ravel(): vn, fp, fn, vp.")

Agora, em texto: escreva 3 a 4 frases respondendo (a) qual limiar você
recomenda para o Clube e por quê, e (b) se valeria a pena construir este
modelo, dado que ligar para todo mundo custa quase o mesmo. Edite esta
célula (duplo clique nela) e escreva sua resposta no lugar deste
parágrafo.